# Super-Resolution Training — Colab Runner

This notebook is a thin wrapper only. All real logic (data pipeline, model, training loop, evaluation) lives in the versioned `.py` files in this repo — nothing is redefined here. This notebook just:
1. Mounts Google Drive (for checkpoint persistence across session drops)
2. Gets the codebase onto the Colab VM (clone from GitHub, or upload/sync from Drive)
3. Installs dependencies
4. Runs `scripts/*.sh` / `train.py` / `evaluate.py` exactly as you would locally

**Before running:** set Runtime -> Change runtime type -> GPU (T4).

## 1. Mount Drive (for checkpoint persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this to wherever you keep the project on Drive.
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sr-project'

## 2. Get the codebase onto the Colab VM

Option A (recommended): push this repo to a GitHub repo (can be private) and clone it.
Option B: if you've already copied the `sr-project` folder into Drive, just `cd` into it directly instead of cloning.

In [ ]:
# --- Option A: clone from GitHub ---
# !git clone https://github.com/<your-username>/sr-project.git /content/sr-project
# %cd /content/sr-project

# --- Option B: use the copy already on Drive ---
%cd {DRIVE_PROJECT_DIR}

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. (First time only) Build train/val/test splits
Skip this if `splits/train.txt` etc. already exist and are committed to the repo/Drive copy.

In [ ]:
!python3 data/prepare_splits.py --source_dir source_images --out_dir splits --train_frac 0.8 --val_frac 0.1 --seed 42

## 5. Train: baseline (naive degradation)
Writes checkpoints to `runs/small_naive/checkpoints/` -- since this run's `--output_dir` lives under the Drive-mounted project folder, checkpoints survive a session disconnect. Use `--resume` to continue an interrupted run.

In [ ]:
!bash scripts/run_baseline_naive.sh

## 6. Train: final model (realistic degradation)

In [ ]:
!bash scripts/run_final_realistic.sh

## 7. Evaluate: baseline vs final, both against a realistic test set

In [ ]:
!bash scripts/run_evaluation.sh

## 8. View results

In [ ]:
import pandas as pd
print(pd.read_csv('eval_results/metrics_table.csv'))

from PIL import Image
Image.open('eval_results/qualitative_comparison.png')

In [ ]:
# View training progression (loss/PSNR curves) for the final model
import pandas as pd
log = pd.read_csv('runs/small_realistic/logs/train_log.csv')
log.plot(x='epoch', y=['train_loss', 'val_loss'], title='Loss curves')
log.plot(x='epoch', y='val_psnr', title='Validation PSNR over training')